In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:

spark = SparkSession.builder \
    .appName("transformation") \
    .config("spark.sql.adaptive.enabled","true") \
    .getOrCreate()

#### Customers

In [0]:
df=spark.table("gizmobox.bronze.v_customers")

In [0]:
df.show()

##### 1. Remove records with null customer id

In [0]:
%sql
select  *
from gizmobox.bronze.v_customers
where customer_id is null;


In [0]:
df=df.filter(col("customer_id").isNotNull())

##### 2. Drop duplicate values based on created_timestamp

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy(col("created_timestamp").desc())

df_dedup = (
    df.withColumn("rn", row_number().over(window_spec))
    .filter(col("rn")==1)
    .drop("rn")
)

In [0]:
display(df_dedup)

In [0]:
df_final = (
    df_dedup
    .withColumn("created_timestamp", col("created_timestamp").cast("timestamp"))
    .withColumn("date_of_birth", col("date_of_birth").cast("date"))
    .withColumn("member_since", col("member_since").cast("date"))
)

In [0]:
display(df_final)

In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gizmobox.silver.customers")

In [0]:
display(spark.sql("select * from gizmobox.silver.customers"))

In [0]:
_schema = f"""
    payment_id integer,
    order_id integer,
    payment_timestamp timestamp,
    payment_status integer,
    payment_method string
"""

payments = spark.read.format("csv").schema(_schema).load("/Volumes/gizmobox/landing/operational_data/payments")

In [0]:
display(payments)

In [0]:
# Check null values

null_counts = (
    payments.select([sum(col(c).isNull().cast("int")).alias(c)
                     for c in payments.columns])
    .collect()[0]
    .asDict()

)
for column, count in null_counts.items():
    if count > 0:
        print(f"{column}: {count}")

In [0]:
payments.createOrReplaceTempView("payments")

In [0]:
payment_silver = spark.sql("""
SELECT 
    payment_id,
    order_id,
    cast(date_format(payment_timestamp, 'yyyy-MM-dd') as date) as payment_date,
    payment_status,
    payment_method
FROM payments
""")


In [0]:
display(payment_silver)

In [0]:
payment_silver = spark.sql("""
select 
   payment_id,
    order_id,
    cast(date_format(payment_timestamp, 'yyyy-MM-dd') as date) as payment_date,
    date_format(payment_timestamp, 'HH:mm:ss') as payment_time,
    case payment_status
        when 1 then 'success'
        when 2 then 'pending'
        when 3 then 'failed'
        when 4 then 'cancelled'
    end as payment_status,
    payment_method
from payments
""")

In [0]:
display(payment_silver)

In [0]:
payment_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gizmobox.silver.payments")

In [0]:
spark.sql("select * from gizmobox.silver.payments").show()

In [0]:
refund = spark.sql("select * from gizmobox.bronze.v_refunds")

In [0]:
refund_silver = spark.sql("""
select 
    refund_id,
    payment_id,
    cast(date_format(refund_timestamp, 'yyyy-MM-dd') as date) as refund_date,
    date_format(refund_timestamp, 'HH:mm:ss') as refund_time,
    refund_amount,
    split(refund_reason,':')[0] as refund_reason,
    split(refund_reason,':')[1] as refund_source
from gizmobox.bronze.refunds
""")

In [0]:
display(refund_silver)

In [0]:
refund_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gizmobox.silver.refunds")

In [0]:
%sql
select * from gizmobox.silver.refunds;

##### Transform Memberships data

In [0]:
display(spark.sql("select path from gizmobox.bronze.v_memberships"))

In [0]:
membership_silver = spark.sql("""
    select 
        regexp_extract(path, '.*/([0-9]+)\\.png$',1) as customer_id,
        content as membership_card 
from gizmobox.bronze.v_memberships;
""")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox.silver.memberships
AS 
select 
        regexp_extract(path, '.*/([0-9]+)\\.png$',1) as customer_id,
        content as membership_card 
from gizmobox.bronze.v_memberships;


In [0]:
%sql
select * from gizmobox.silver.memberships;

##### Transform Addresses data

In [0]:
display(spark.sql("select * from gizmobox.bronze.v_addresses"))

In [0]:

add_silver = spark.sql("""
select * 
from(
select 
    customer_id,
    address_type,
    address_line_1,
    city,
    state,
    postcode
from gizmobox.bronze.v_addresses
)

pivot (
    max(address_line_1) as address_line_1,
    max(city) as city,
    max(state) as state,
    max(postcode) as postcode
    for address_type in ('shipping', 'billing')
)
"""
)



In [0]:
add_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gizmobox.silver.addresses")

In [0]:
%sql
select * from gizmobox.silver.addresses

#### Transform Orders data

In [0]:
%sql

select * from gizmobox.bronze.v_orders;

##### Extract top level object value


In [0]:
%sql
select 
    value:order_id,
    value
from gizmobox.bronze.v_orders

##### Extract arrays

In [0]:
%sql
select 
    value:items[0] as item_1,
    value:items[1] as item_2,
    value
from gizmobox.bronze.v_orders

##### CAST column values to a specific data type

In [0]:
%sql
select * from gizmobox.bronze.v_orders

In [0]:
%sql
select value:items[0].item_id::INTEGER as item_1_item_id,
    value:items[0] as item_1,
    value 
from gizmobox.bronze.v_orders

#### Convert JSON String to JSON Object

##### 1. Pre-process JSON using regexp_replace function

In [0]:
%sql
CREATE or REPLACE TEMP VIEW tv_orders_fixed AS
select
    value,
    regexp_replace(value, '"order_date: (\\d{4}-\\d{2}-\\d{2})', '"order_date": ":1"') as fixed_value
from gizmobox.bronze.v_orders
    


### Transform JSON string to JSON object

In [0]:
%sql
select fixed_value

from tv_orders_fixed;

In [0]:
%sql
select
    fixed_value,
    schema_of_json(fixed_value)
from tv_orders_fixed
limit 1;

In [0]:
%sql
select from_json(fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value,
fixed_value
from tv_orders_fixed


In [0]:
%sql
DROP TABLE IF EXISTS gizmobox.silver.orders_json;
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders_json AS
select from_json(fixed_value, 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value
from tv_orders_fixed


In [0]:
%sql
select * from gizmobox.silver.orders_json limit 10;